# StockEcho 한국어 BERTopic 실험

한국어 SentenceTransformer 임베딩, Kiwi c-TF-IDF, 날짜별 Event 분리와 7→14→30일 Top 3 선정을 실행합니다. Colab에서 GPU runtime을 선택하면 자동으로 CUDA를 사용합니다.

In [ ]:
from pathlib import Path

repo = Path('/content/StockEcho')
if not repo.exists():
    !git clone https://github.com/kimminjunnn/StockEcho.git /content/StockEcho
%cd /content/StockEcho
!pip install -q -r requirements-bertopic.txt

아래 셀에서 `005930_articles.jsonl` 또는 `000660_articles.jsonl`을 업로드합니다. 파일명에서 종목 코드를 읽습니다.

In [ ]:
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('한 번에 종목별 JSONL 파일 하나를 업로드하세요.')
filename, content = next(iter(uploaded.items()))
stock_code = filename.split('_', 1)[0]
input_dir = Path('data/processed/bertopic')
input_dir.mkdir(parents=True, exist_ok=True)
input_path = input_dir / f'{stock_code}_articles.jsonl'
input_path.write_bytes(content)
print(stock_code, input_path, f'{len(content):,} bytes')

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
output_dir = Path('/content/bertopic-results')
!python -m collector.jobs.run_bertopic --stock-code {stock_code} --input {input_path} --output-dir {output_dir} --device {device}

In [ ]:
import shutil

archive = shutil.make_archive('/content/stockecho-bertopic-results', 'zip', output_dir)
files.download(archive)